# 10 — Aberturas, encerramentos e saldo (Excel)

**Objetivo:** gerar uma planilha Excel com a quantidade de aberturas, de encerramentos e o saldo de estabelecimentos, por mês/ano e por ano, para a análise da dinâmica de negócios do município.

**Entradas:** `df_cnpj_espacial` (08), colunas `data_abertura`, `data_encerramento`, `data_situacao` e `data_inicio_atividade`.

**Saídas (`04_entrega/`):** `aberturas_encerramentos_saldo.xlsx`, com duas abas: `por_mes_ano` e `por_ano` (um período por coluna).

**Como rodar:** de cima para baixo. Não precisa do 09. **Dados internos da Prefeitura:** `04_entrega/` fica fora do Git.

**Definições:**
- **Aberturas:** estabelecimentos (matriz e filial, cada CNPJ de 14 dígitos) com data de início da atividade no período.
- **Encerramentos:** estabelecimentos com situação **Baixada**, na data da baixa.
- **Saldo:** quantos estabelecimentos permanecem abertos no **fim do período** = abertos até a data (em qualquer ano) menos baixados até a data. Suspensos, inaptos e nulos contam como abertos.
- O **último período** vai até a **data da base**: a maior data dos registros da Receita (ex.: 15/11/2026).

**Limite do método:** o saldo é reconstruído a partir da base atual da Receita. Estabelecimentos que saíram do município ou deixaram de constar na base não aparecem; por isso o saldo dos anos antigos tende a ficar menor que o real. Aberturas e encerramentos por período sofrem menos com isso.

**Seções:** 1. Conferência das datas · 2. Cálculo das tabelas · 3. Gravação do Excel · 4. Resumo


In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, saldo, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("10_exportar_aberturas_encerramentos", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== PARÂMETROS (config.yaml) =====
ARQ_SALDO = cfg.caminho("entrega.saldo")                    # 04_entrega/aberturas_encerramentos_saldo.xlsx
ANO_INICIAL = cfg.parametro("saldo.ano_inicial")            # primeiro ano mostrado (vazio = desde o primeiro registro)
SITUACAO_BAIXADA = cfg.parametro("legendas_receita.situacao_baixada")   # código da situação que conta como encerramento

# ===== ENTRADAS =====
df_cnpj_espacial = fontes.ler_processamento(cfg, "df_cnpj_espacial")   # 08

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")


## 1. Conferência das datas
Confere as colunas de data, define a **data da base** (maior data entre `data_situacao` e `data_inicio_atividade`, a mesma regra do arquivo de informações do Monitor) e conta os casos que não entram no cálculo: estabelecimentos sem data de abertura, baixados sem data da baixa e encerramentos anteriores à abertura. Só logs.


In [ ]:
log.info("SEÇÃO 1: conferindo as datas...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_s1 = time.time()

# ===================================================================
# ETAPA 1: CONFERIR AS COLUNAS
# ===================================================================
log.info("Etapa 1: conferindo as colunas necessárias...")
colunas_necessarias = ["cnpj", "situacao_cadastral", "data_situacao", "data_inicio_atividade", "data_abertura", "data_encerramento"]
faltam = [c for c in colunas_necessarias if c not in df_cnpj_espacial.columns]
if faltam:
    raise ValueError(f"Seção interrompida: faltam colunas no df_cnpj_espacial: {faltam}. Rode de novo o notebook 08.")
# As datas precisam ser datas de verdade (o notebook 08 faz a conversão).
for coluna_data in ["data_abertura", "data_encerramento", "data_situacao", "data_inicio_atividade"]:
    if not pd.api.types.is_datetime64_any_dtype(df_cnpj_espacial[coluna_data]):
        raise TypeError(f"Seção interrompida: '{coluna_data}' não é uma coluna de datas. Rode de novo o notebook 08.")
abertura = df_cnpj_espacial["data_abertura"]
encerramento = df_cnpj_espacial["data_encerramento"]
log.info(f"Etapa 1 concluída! {len(df_cnpj_espacial):,} estabelecimentos (matriz e filial).")

# ===================================================================
# ETAPA 2: DATA DA BASE E CASOS FORA DO CÁLCULO
# ===================================================================
log.info("\nEtapa 2: definindo a data da base e contando os casos fora do cálculo...")
# Data da base: a última data com registro (o último período termina nela).
datas_todas = pd.concat([df_cnpj_espacial["data_situacao"], df_cnpj_espacial["data_inicio_atividade"]]).dropna()
DATA_BASE = datas_todas.max()
baixada = pd.to_numeric(df_cnpj_espacial["situacao_cadastral"], errors="coerce") == SITUACAO_BAIXADA

sem_abertura = int(abertura.isna().sum())
baixadas_sem_data = int((baixada & encerramento.isna()).sum())
antes_da_abertura = int((encerramento < abertura).sum())
log.info(f"    Data da base (última data com registro): {DATA_BASE:%d/%m/%Y}")
log.info(f"    Primeira abertura: {abertura.min():%d/%m/%Y} | última: {abertura.max():%d/%m/%Y}")
log.info(f"    Sem data de abertura (ficam fora do saldo e das aberturas): {sem_abertura:,}")
log.info(f"    Baixados sem data da baixa (ficam fora dos encerramentos e do saldo): {baixadas_sem_data:,}")
log.info(f"    Encerramento anterior à abertura (data com erro; no saldo vale a data da abertura): {antes_da_abertura:,}")
if ANO_INICIAL:
    log.info(f"    Primeiro ano mostrado (config.yaml): {ANO_INICIAL}")
log.info("Etapa 2 concluída!")

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_s1)}.")


## 2. Cálculo das tabelas
Monta as duas tabelas (`saldo.tabela_por_periodo`, em `scripts/src/saldo.py`): por mês/ano e por ano, com as linhas **Aberturas**, **Encerramentos** e **Saldo (estabelecimentos abertos no fim do período)**. Confere que o saldo do último período é igual a "estabelecimentos com abertura menos baixados com data" e que a soma das aberturas e dos encerramentos bate com o total. Só logs.


In [ ]:
log.info("SEÇÃO 2: calculando as tabelas de aberturas, encerramentos e saldo...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_s2 = time.time()

# ===================================================================
# ETAPA 1: CALCULAR AS DUAS TABELAS
# ===================================================================
log.info("Etapa 1: calculando por mês/ano e por ano...")
tabela_mes = saldo.tabela_por_periodo(abertura, encerramento, DATA_BASE, "M", ANO_INICIAL)
tabela_ano = saldo.tabela_por_periodo(abertura, encerramento, DATA_BASE, "Y", ANO_INICIAL)
log.info(f"Etapa 1 concluída! Por mês/ano: {tabela_mes.shape[1]:,} períodos | por ano: {tabela_ano.shape[1]:,} períodos.")

# ===================================================================
# ETAPA 2: CONFERIR AS TABELAS
# ===================================================================
log.info("\nEtapa 2: conferindo as tabelas...")
# Saldo esperado no fim: quem tem data de abertura menos quem tem data de abertura E de baixa.
saldo_esperado = int(abertura.notna().sum()) - int((abertura.notna() & encerramento.notna()).sum())
for nome_tabela, tabela in [("por mês/ano", tabela_mes), ("por ano", tabela_ano)]:
    saldo_final = int(tabela.loc[saldo.ROTULO_SALDO].iloc[-1])
    log.info(f"    Saldo no último período ({nome_tabela}): {saldo_final:,} (esperado: {saldo_esperado:,}): "
             f"{'confere' if saldo_final == saldo_esperado else 'NÃO confere'}")
    # Se o ano inicial foi pedido, a soma só cobre os períodos mostrados; sem ele, tem que bater com o total.
    if not ANO_INICIAL:
        soma_ab = int(tabela.loc[saldo.ROTULO_ABERTURAS].sum())
        soma_en = int(tabela.loc[saldo.ROTULO_ENCERRAMENTOS].sum())
        log.info(f"    Soma das aberturas ({nome_tabela}): {soma_ab:,} (esperado: {int(abertura.notna().sum()):,}): "
                 f"{'confere' if soma_ab == int(abertura.notna().sum()) else 'NÃO confere'}")
        log.info(f"    Soma dos encerramentos ({nome_tabela}): {soma_en:,} (esperado: {int(encerramento.notna().sum()):,}): "
                 f"{'confere' if soma_en == int(encerramento.notna().sum()) else 'NÃO confere'}")
    negativos = int((tabela.loc[saldo.ROTULO_SALDO] < 0).sum())
    (log.info if negativos == 0 else log.warning)(f"    Períodos com saldo negativo ({nome_tabela}): {negativos:,}")
log.info("Etapa 2 concluída!")

log.info("\n--- ÚLTIMOS 6 ANOS (por ano) ---")
for rotulo_periodo in tabela_ano.columns[-6:]:
    ab = int(tabela_ano.at[saldo.ROTULO_ABERTURAS, rotulo_periodo])
    en = int(tabela_ano.at[saldo.ROTULO_ENCERRAMENTOS, rotulo_periodo])
    sd = int(tabela_ano.at[saldo.ROTULO_SALDO, rotulo_periodo])
    log.info(f"  {rotulo_periodo}: aberturas {ab:,} | encerramentos {en:,} | saldo {sd:,}")

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_s2)}.")


## 3. Gravação do Excel
Grava `04_entrega/aberturas_encerramentos_saldo.xlsx` com duas abas (`por_mes_ano` e `por_ano`): as medidas nas linhas e um período em cada coluna, do mais antigo ao mais novo (o último período mostra a data da base no título). Abaixo de cada tabela ficam notas com as definições. Depois de gravar, reabre o arquivo e compara os números com as tabelas. Exige `xlsxwriter` e `openpyxl`. Só logs.


In [ ]:
log.info("SEÇÃO 3: gravando o Excel...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_s3 = time.time()

# ===================================================================
# ETAPA 1: VERIFICAR AS BIBLIOTECAS E O ARQUIVO
# ===================================================================
log.info("Etapa 1: verificando as bibliotecas e o arquivo de saída...")
for nome_biblioteca in ["xlsxwriter", "openpyxl"]:
    if importlib.util.find_spec(nome_biblioteca) is None:
        raise ImportError(f"Seção interrompida: falta a biblioteca {nome_biblioteca} (pip install {nome_biblioteca}). Nada foi gravado.")
import xlsxwriter
import openpyxl

ARQUIVO_SAIDA = Path(ARQ_SALDO).resolve()
ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
# Se já existir um arquivo com esse nome (de uma rodada anterior), ele é apagado para ser recriado do zero.
if ARQUIVO_SAIDA.exists():
    try:
        ARQUIVO_SAIDA.unlink()
    except PermissionError:
        # O Windows não deixa apagar um arquivo que está aberto no Excel.
        raise PermissionError(f"Seção interrompida: não foi possível apagar {ARQUIVO_SAIDA}. Feche o arquivo no Excel e rode de novo.")
log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: GRAVAR AS DUAS ABAS
# ===================================================================
log.info(f"\nEtapa 2: gravando {ARQUIVO_SAIDA}...")
workbook = xlsxwriter.Workbook(str(ARQUIVO_SAIDA))
formato_titulo = workbook.add_format({"bold": True, "align": "center", "valign": "vcenter", "text_wrap": True,
                                      "bg_color": "#D9E1F2", "border": 1})
formato_linha = workbook.add_format({"bold": True, "border": 1})
formato_numero = workbook.add_format({"num_format": "#,##0", "border": 1})
formato_nota = workbook.add_format({"italic": True, "font_color": "#555555"})

# Notas escritas embaixo de cada tabela.
notas = [
    "Aberturas: estabelecimentos (matriz e filial) com data de início da atividade no período.",
    "Encerramentos: estabelecimentos com situação Baixada, na data da baixa.",
    "Saldo: estabelecimentos abertos no fim do período = abertos até a data (em qualquer ano) menos baixados até a data. Suspensos, inaptos e nulos contam como abertos.",
    f"O último período vai até a data da base ({DATA_BASE:%d/%m/%Y}), a última data com registro na Receita Federal.",
    f"Ficam fora: {sem_abertura:,} estabelecimentos sem data de abertura e {baixadas_sem_data:,} baixados sem data da baixa.",
    "O saldo vem da base atual da Receita: quem saiu do município ou deixou de constar não aparece, então o saldo dos anos antigos tende a ficar menor que o real.",
]

for nome_aba, tabela in [("por_mes_ano", tabela_mes), ("por_ano", tabela_ano)]:
    aba = workbook.add_worksheet(nome_aba)
    # Cabeçalho: "Indicador" e um título por período.
    aba.write(0, 0, "Indicador", formato_titulo)
    aba.write_row(0, 1, list(tabela.columns), formato_titulo)
    aba.set_row(0, 32)
    # Uma linha por medida, com os números em cada período.
    for numero_linha, (rotulo_linha, valores) in enumerate(tabela.iterrows(), start=1):
        aba.write(numero_linha, 0, rotulo_linha, formato_linha)
        aba.write_row(numero_linha, 1, [int(v) for v in valores], formato_numero)
    # Larguras: a 1ª coluna é larga (nomes das linhas) e a última leva a data da base no título.
    aba.set_column(0, 0, 52)
    aba.set_column(1, tabela.shape[1] - 1, 11)
    aba.set_column(tabela.shape[1], tabela.shape[1], 26)
    # Congela o cabeçalho e a 1ª coluna para rolar a tabela sem perder os títulos.
    aba.freeze_panes(1, 1)
    # Notas embaixo da tabela.
    for numero_nota, nota in enumerate(notas):
        aba.write(len(tabela) + 2 + numero_nota, 0, nota, formato_nota)
    log.info(f"    Aba {nome_aba}: {tabela.shape[1]:,} períodos.")

try:
    workbook.close()
except (PermissionError, xlsxwriter.exceptions.FileCreateError):
    raise PermissionError(f"Seção interrompida: não foi possível gravar {ARQUIVO_SAIDA}. "
                          "Verifique se o arquivo não está aberto no Excel e se há permissão de escrita na pasta.")
log.info(f"Etapa 2 concluída! Arquivo gravado ({ARQUIVO_SAIDA.stat().st_size / 1e3:,.0f} KB).")

# ===================================================================
# ETAPA 3: REABRIR E CONFERIR
# ===================================================================
log.info("\nEtapa 3: reabrindo o Excel para conferir...")
livro = openpyxl.load_workbook(ARQUIVO_SAIDA, read_only=True)
log.info(f"    Abas: {livro.sheetnames}: {'confere' if livro.sheetnames == ['por_mes_ano', 'por_ano'] else 'NÃO confere'}")
for nome_aba, tabela in [("por_mes_ano", tabela_mes), ("por_ano", tabela_ano)]:
    linhas_lidas = list(livro[nome_aba].iter_rows(min_row=1, max_row=len(tabela) + 1, values_only=True))
    titulos = list(linhas_lidas[0][1:])
    numeros = [list(l[1:]) for l in linhas_lidas[1:]]
    iguais = titulos == list(tabela.columns) and numeros == tabela.values.tolist()
    log.info(f"    Aba {nome_aba}: títulos e números iguais às tabelas: {'confere' if iguais else 'NÃO confere'}")
livro.close()
log.info("Etapa 3 concluída!")

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_s3)}.")


## 4. Resumo
Números principais, arquivo gerado e tempo total.


In [ ]:
log.info("RESUMO do 10_exportar_aberturas_encerramentos...")
log.info(f"Data da base: {DATA_BASE:%d/%m/%Y} | períodos por mês/ano: {tabela_mes.shape[1]:,} | por ano: {tabela_ano.shape[1]:,}")
log.info(f"Estabelecimentos abertos na data da base (saldo): {int(tabela_ano.loc[saldo.ROTULO_SALDO].iloc[-1]):,}")
log.info("Arquivo gerado:")
log.info(f"    {Path(ARQ_SALDO).resolve().relative_to(RAIZ)} ({Path(ARQ_SALDO).stat().st_size / 1e3:,.0f} KB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")
